In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_raw_customers = pd.read_csv("../data/raw/customers.csv")
df_raw_marketing = pd.read_csv("../data/raw/marketing.csv")
df_raw_order_items = pd.read_csv("../data/raw/order_items.csv")
df_raw_orders = pd.read_csv("../data/raw/orders.csv")
df_raw_products = pd.read_csv("../data/raw/products.csv")
df_raw_website_sessions = pd.read_csv("../data/raw/website_sessions.csv")

In [3]:
raw_datasets = {
    "customers": df_raw_customers,
    "marketing": df_raw_marketing,
    "order_items": df_raw_order_items,
    "orders": df_raw_orders,
    "products": df_raw_products,
    "website_sessions": df_raw_website_sessions
}

In [4]:
i = 0

for name, df in raw_datasets.items():
    print(f"\n{'=' * 20} {name.upper()} {'=' * 20}")
    df.info()


==================== CUSTOMERS ====================
<class 'pandas.DataFrame'>
RangeIndex: 53000 entries, 0 to 52999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   customer_id         53000 non-null  str  
 1   signup_date         53000 non-null  str  
 2   region              53000 non-null  str  
 3   country             53000 non-null  str  
 4   acquisition_source  53000 non-null  str  
 5   customer_type       53000 non-null  str  
 6   email_opt_in        53000 non-null  str  
dtypes: str(7)
memory usage: 2.8 MB

==================== MARKETING ====================
<class 'pandas.DataFrame'>
RangeIndex: 2190 entries, 0 to 2189
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            2190 non-null   str    
 1   channel         2190 non-null   str    
 2   spend_eur       2190 non-null   float64
 3   campaign_coun

In [5]:
df_raw_customers.head()

,customer_id,signup_date,region,country,acquisition_source,customer_type,email_opt_in
0,C000001,2023-01-25,Lower Saxony,DE,direct,consumer,no
1,C000002,2023-01-29,Bavaria,AT,organic,small_business,yes
2,C000003,2022-08-01,Saxony,DE,organic,small_business,yes
3,C000004,2022-09-25,North Rhine-Westphalia,DE,paid_social,small_business,no
4,C000005,2023-09-28,North Rhine-Westphalia,DE,paid_search,small_business,yes


In [6]:
df_raw_customers["customer_id"].nunique()

53000

In [7]:
df_raw_customers["region"].value_counts()

region
North Rhine-Westphalia    11749
Bavaria                   10032
Baden-Württemberg          8370
Hesse                      6460
Lower Saxony               5319
Berlin                     4190
Saxony                     3734
Hamburg                    3146
Name: count, dtype: int64

In [8]:
df_raw_customers["country"].value_counts()

country
DE    42365
AT    10635
Name: count, dtype: int64

In [16]:
print("--- Regions assigned to 'AT' as country ---")
df_raw_customers[df_raw_customers["country"] == "AT"].region.value_counts()

--- Regions assigned to 'AT' as country ---


region
North Rhine-Westphalia    2410
Bavaria                   2013
Baden-Württemberg         1701
Hesse                     1294
Lower Saxony              1063
Berlin                     853
Saxony                     680
Hamburg                    621
Name: count, dtype: int64

In [17]:
df_raw_customers.columns

Index(['customer_id', 'signup_date', 'region', 'country', 'acquisition_source',
       'customer_type', 'email_opt_in'],
      dtype='str')

In [20]:
df_raw_customers["acquisition_source"].value_counts()

acquisition_source
direct         8934
email          8895
paid_social    8818
paid_search    8795
organic        8794
referral       8764
Name: count, dtype: int64

In [21]:
df_raw_customers["customer_type"].value_counts()

customer_type
consumer          39698
small_business    13302
Name: count, dtype: int64

In [24]:
df_raw_customers["email_opt_in"].value_counts()

email_opt_in
yes    39750
no     13250
Name: count, dtype: int64

In [41]:
df_raw_customers['signup_date'].min()

'2021-11-08'

In [40]:
df_raw_customers['signup_date'].max()

'2025-01-01'

Initial Findings — Customers Dataset

Data types: Some columns may require data type conversion. `signup_date` should be converted to a datetime type, while `email_opt_in` should be converted to a Boolean type.

Country–region consistency: There appears to be a potential inconsistency between the country and region columns. Some customers with AT (Austria) as their country are associated with German regions. This requires further investigation.

Signup dates: Customer registration dates range from November 8, 2021, to January 1, 2025. No obvious anomalies were identified in the date range, although further validation may be necessary.

In [42]:
df_raw_marketing.head()

,date,channel,spend_eur,campaign_count
0,2025-01-01,organic_search,86.64,0
1,2025-01-01,paid_search,1759.59,1
2,2025-01-01,paid_social,836.00,1
3,2025-01-01,email,133.06,0
4,2025-01-01,direct,33.21,0


In [99]:
print(f"Earliest date: {df_raw_marketing.date.min()}\nLatest date: {df_raw_marketing.date.max()}")

Earliest date: 2025-01-01
Latest date: 2025-12-31


In [52]:
df_raw_marketing["channel"].value_counts()

channel
organic_search    365
paid_search       365
paid_social       365
email             365
direct            365
referral          365
Name: count, dtype: int64

In [56]:
df_raw_marketing["campaign_count"].value_counts()

campaign_count
0    1460
4     191
3     185
2     185
1     169
Name: count, dtype: int64

In [60]:
df_raw_marketing['spend_eur'].describe()

count    2190.000000
mean      498.747087
std       631.361216
min        31.930000
25%        60.375000
50%       104.550000
75%       940.395000
max      1864.130000
Name: spend_eur, dtype: float64

In [66]:
df_raw_marketing.groupby("channel")[["spend_eur","campaign_count"]].sum()

,spend_eur,campaign_count
channel,,
direct,12885.39,0
email,44091.75,0
organic_search,33249.38,0
paid_search,600223.47,909
paid_social,379762.25,949
referral,22043.88,0


In [67]:
df_marketing_summary = df_raw_marketing.groupby("channel")[["spend_eur","campaign_count"]].sum()
df_marketing_summary["spend_eur_per_campaign"] = (
    df_marketing_summary["spend_eur"] /
    df_marketing_summary["campaign_count"]
)

df_marketing_summary

,spend_eur,campaign_count,spend_eur_per_campaign
channel,,,
direct,12885.39,0,inf
email,44091.75,0,inf
organic_search,33249.38,0,inf
paid_search,600223.47,909,660.311848
paid_social,379762.25,949,400.170969
referral,22043.88,0,inf


In [73]:
df_raw_marketing.duplicated().sum()

np.int64(0)

In [74]:
df_raw_marketing.duplicated(
    subset=["date", "channel"]
).sum()

np.int64(0)

Initial Findings — Marketing Dataset

- **Missing values:** No missing values were identified.
- **Duplicate records:** No exact duplicate rows were found.
- **Record uniqueness:** Each combination of `date` and `channel` is unique, consistent with the expected daily, channel-level granularity.
- **Marketing expenditure:** `spend_eur` ranges from €31.93 to €1,864.13. All values are positive, with no obvious anomalies identified.
- **Campaign counts:** `campaign_count` ranges from 0 to 4, with no obviously implausible values.
- **Open business questions:**
  - Clarify why marketing expenditure is reported for channels such as `direct` and `email`, despite having no recorded campaigns.
  - Confirm whether `campaign_count` represents **campaigns active on a given day** or **newly launched campaigns**. This distinction is essential to avoid double-counting campaigns and incorrectly interpreting campaign-level spending metrics.

In [75]:
df_raw_orders.head()

,order_id,customer_id,order_timestamp,channel,device,order_status,order_discount_pct,shipping_method,shipping_country
0,O0000001,C007960,2025-01-01 13:59:00,organic_search,desktop,completed,10,standard,AT
1,O0000002,C022927,2025-01-01 14:18:00,direct,mobile,completed,5,standard,DE
2,O0000003,C030260,2025-01-01 09:31:00,paid_social,desktop,completed,5,standard,DE
3,O0000004,C029113,2025-01-01 15:55:00,paid_social,desktop,completed,5,express,DE
4,O0000005,C041076,2025-01-01 10:03:00,direct,desktop,completed,0,standard,DE


In [80]:
df_raw_orders.shape

(150150, 9)

In [81]:
df_raw_orders['order_id'].nunique()

150000

There are rows that are not unique in order id.

In [86]:
df_raw_orders[df_raw_orders.duplicated(keep=False)].sort_values(by="order_id")

,order_id,customer_id,order_timestamp,channel,device,order_status,order_discount_pct,shipping_method,shipping_country
1962,O0001963,C047159,2025-01-05 09:40:00,direct,mobile,completed,10,standard,DE
150054,O0001963,C047159,2025-01-05 09:40:00,direct,mobile,completed,10,standard,DE
150007,O0002074,C029832,2025-01-06 06:05:00,paid_social,desktop,completed,0,standard,AT
2073,O0002074,C029832,2025-01-06 06:05:00,paid_social,desktop,completed,0,standard,AT
3351,O0003352,C043585,2025-01-09 05:58:00,direct,mobile,completed,0,standard,DE
...,...,...,...,...,...,...,...,...,...
150031,O0147574,C017123,2025-12-26 02:36:00,email,desktop,cancelled,0,express,DE
150091,O0148155,C012667,2025-12-27 23:27:00,email,mobile,cancelled,0,express,DE
148154,O0148155,C012667,2025-12-27 23:27:00,email,mobile,cancelled,0,express,DE
150120,O0149771,C029580,2025-12-31 16:34:00,email,mobile,completed,10,express,DE


In [ ]:
df_raw_orders[df_raw_orders.duplicated(keep=False)].sort_values(by="order_id")

,order_id,customer_id,order_timestamp,channel,device,order_status,order_discount_pct,shipping_method,shipping_country
150126,O0146375,C044403,2025-12-23 23:15:00,email,desktop,completed,0,express,DE
146374,O0146375,C044403,2025-12-23 23:15:00,email,desktop,completed,0,express,DE
150083,O0147513,C016725,2025-12-25 19:33:00,paid_social,mobile,completed,0,express,AT
147512,O0147513,C016725,2025-12-25 19:33:00,paid_social,mobile,completed,0,express,AT
147573,O0147574,C017123,2025-12-26 02:36:00,email,desktop,cancelled,0,express,DE
150031,O0147574,C017123,2025-12-26 02:36:00,email,desktop,cancelled,0,express,DE
150091,O0148155,C012667,2025-12-27 23:27:00,email,mobile,cancelled,0,express,DE
148154,O0148155,C012667,2025-12-27 23:27:00,email,mobile,cancelled,0,express,DE
150120,O0149771,C029580,2025-12-31 16:34:00,email,mobile,completed,10,express,DE
149770,O0149771,C029580,2025-12-31 16:34:00,email,mobile,completed,10,express,DE


The duplicated order_id row has also the entire row duplicated. It means it is not just a cancellation for the same order because the order_status also does not changes.

In [88]:
df_raw_orders["customer_id"].nunique()

49833

In [89]:
df_raw_orders["customer_id"].value_counts()

customer_id
C009108    12
C032324    12
C004432    11
C049020    11
C040422    11
           ..
C039451     1
C048983     1
C043718     1
C002613     1
C002320     1
Name: count, Length: 49833, dtype: int64

In [90]:
df_raw_orders['channel'].value_counts()

channel
paid_search       37306
organic_search    34469
direct            25509
paid_social       23793
email             19426
referral           9146
ORGANIC_SEARCH       58
PAID_SEARCH          51
PAID_SOCIAL          43
DIRECT               42
EMAIL                32
REFERRAL             18
Name: count, dtype: int64

Same category, but one lowercase and the other uppercase. Data should be treated

In [91]:
df_raw_orders['device'].value_counts()

device
mobile     85510
desktop    55519
tablet      9121
Name: count, dtype: int64

In [92]:
df_raw_orders['order_status'].value_counts()

order_status
completed    136178
cancelled      7684
returned       6288
Name: count, dtype: int64

In [93]:
df_raw_orders['shipping_method'].value_counts()

shipping_method
express     75242
standard    74908
Name: count, dtype: int64

In [94]:
df_raw_orders['shipping_country'].value_counts()

shipping_country
DE    112286
AT     37864
Name: count, dtype: int64

In [95]:
df_raw_orders['order_discount_pct'].value_counts()

order_discount_pct
0     82402
5     27156
10    24010
15    11964
20     4618
Name: count, dtype: int64

In [98]:
df_raw_orders["order_timestamp"].min()
print(f"Earliest date: {df_raw_orders["order_timestamp"].min()}\nLatest date: {df_raw_orders["order_timestamp"].max()}")

Earliest date: 2025-01-01 00:03:00
Latest date: 2025-12-31 23:59:00


In [100]:
df_raw_orders.isna().sum()

order_id                0
customer_id             0
order_timestamp         0
channel               257
device                  0
order_status            0
order_discount_pct      0
shipping_method         0
shipping_country        0
dtype: int64

In [103]:
df_raw_orders.isna()

,order_id,customer_id,order_timestamp,channel,device,order_status,order_discount_pct,shipping_method,shipping_country
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
150145,False,False,False,False,False,False,False,False,False
150146,False,False,False,False,False,False,False,False,False
150147,False,False,False,False,False,False,False,False,False
150148,False,False,False,False,False,False,False,False,False


In [109]:
df_raw_orders[df_raw_orders.isna().any(axis=1)].sample(10)

,order_id,customer_id,order_timestamp,channel,device,order_status,order_discount_pct,shipping_method,shipping_country
97234,O0097235,C040454,2025-08-22 11:10:00,NaN,mobile,completed,15,standard,DE
24444,O0024445,C040403,2025-03-04 13:30:00,NaN,tablet,completed,0,standard,AT
92776,O0092777,C012932,2025-08-11 18:21:00,NaN,desktop,completed,0,standard,DE
105900,O0105901,C037335,2025-09-11 07:55:00,NaN,mobile,completed,5,standard,DE
67258,O0067259,C007219,2025-06-12 19:16:00,NaN,mobile,completed,0,express,DE
120156,O0120157,C014303,2025-10-17 17:18:00,NaN,mobile,completed,0,standard,DE
139746,O0139747,C039844,2025-12-06 04:19:00,NaN,mobile,completed,10,standard,AT
36092,O0036093,C043583,2025-03-31 21:43:00,NaN,mobile,completed,0,express,DE
143866,O0143867,C028708,2025-12-17 09:20:00,NaN,tablet,completed,5,standard,DE
130272,O0130273,C012909,2025-11-13 04:50:00,NaN,mobile,completed,0,express,AT


There are NaN values in the `channel` column and there seems to be no specific pattern or reason for it. Its not related to cancelled transations, or devices etc.

Initial Findings — Orders Dataset

**1. Missing values**
- The `channel` column contains missing values.
- No clear relationship was identified between these missing values and order status or device type.
- The underlying cause remains unknown and requires further investigation.

**2. Inconsistent categorical values**
- The `channel` column contains the same category represented with different capitalization.
- These values should be standardized during data cleaning to avoid treating the same channel as separate categories.

**3. Duplicate records**
- Some `order_id` values appear more than once.
- These records are exact duplicates, including their order statuses.
- Duplicate records could inflate order counts and financial metrics if not handled appropriately.

**4. Date and numerical validation**
- No unexpected dates or order status categories were identified.
- No negative monetary amounts or percentages were found.
- Numerical values appear reasonable based on the initial exploratory analysis.

**5. Customer ID validation**
- The number of unique customer IDs in the orders dataset is lower than in the customers dataset, which is plausible because not every registered customer necessarily places an order.
- Referential integrity between `orders.csv` and `customers.csv` has not yet been verified and will be assessed during cross-dataset validation.

In [120]:
df_raw_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      600 non-null    str    
 1   product_name    600 non-null    str    
 2   category        600 non-null    str    
 3   brand           600 non-null    str    
 4   list_price_eur  600 non-null    float64
 5   unit_cost_eur   600 non-null    float64
 6   launch_date     600 non-null    str    
 7   active_flag     600 non-null    int64  
dtypes: float64(2), int64(1), str(5)
memory usage: 37.6 KB


In [121]:
df_raw_products.head()

,product_id,product_name,category,brand,list_price_eur,unit_cost_eur,launch_date,active_flag
0,P0001,Vela Electronics Item 001,Electronics,Vela,71.82,39.35,2024-01-01,1
1,P0002,Karo Home Item 002,Home & Kitchen,Karo,74.66,46.53,2024-01-01,1
2,P0003,Meridian Sports Item 003,Sports & Outdoors,Meridian,70.02,34.15,2024-01-01,1
3,P0004,Avento Beauty Item 004,Beauty & Personal Care,Avento,31.15,15.18,2024-01-01,1
4,P0005,Luma Fashion Item 005,Fashion,Luma,74.90,35.65,2024-01-01,1


In [122]:
df_raw_products["product_id"].nunique()

600

In [125]:
df_raw_products.sample(10)

,product_id,product_name,category,brand,list_price_eur,unit_cost_eur,launch_date,active_flag
97,P0098,Karo Home Item 098,Home & Kitchen,Karo,100.14,66.44,2024-01-01,1
449,P0450,Karo Office Item 450,Office & Stationery,Karo,19.87,10.25,2024-01-01,1
410,P0411,Meridian Sports Item 411,Sports & Outdoors,Meridian,56.62,34.53,2024-01-01,1
440,P0441,Vela Sports Item 441,Sports & Outdoors,Vela,52.39,36.40,2024-01-01,1
265,P0266,Karo Home Item 266,Home & Kitchen,Karo,48.58,32.26,2024-01-01,1
286,P0287,Nova Fashion Item 287,Fashion,Nova,29.45,17.75,2024-01-01,1
19,P0020,Avento Home Item 020,Home & Kitchen,Avento,97.83,45.55,2024-01-01,1
115,P0116,Avento Home Item 116,Home & Kitchen,Avento,32.74,15.96,2024-01-01,1
558,P0559,Nova Electronics Item 559,Electronics,Nova,142.39,76.72,2024-01-01,0
340,P0341,Luma Fashion Item 341,Fashion,Luma,22.87,12.22,2024-01-01,1


Every `product_name` has a name and the text item XXX, XXX is a number that matches the end of the `product_id` number. I noticed that some products have the same name and brand, but differs by the number or the item. They also have difference in price, for example P0020 and P0116. I would only check if they are different products or a duplicated input with different values.

In [126]:
df_raw_products["category"].value_counts()

category
Electronics               100
Home & Kitchen            100
Sports & Outdoors         100
Beauty & Personal Care    100
Fashion                   100
Office & Stationery       100
Name: count, dtype: int64

In [127]:
df_raw_products["brand"].value_counts()

brand
Vela        75
Karo        75
Meridian    75
Avento      75
Luma        75
Hanse       75
Nova        75
Nordline    75
Name: count, dtype: int64

In [130]:
df_raw_products[['list_price_eur', 'unit_cost_eur']].describe()

,list_price_eur,unit_cost_eur
count,600.000000,600.000000
mean,64.292017,36.988250
std,44.003706,25.750508
min,9.380000,4.420000
25%,30.790000,17.545000
50%,54.475000,30.385000
75%,87.585000,50.677500
max,209.120000,136.830000


In [131]:
df_raw_products['product_name'].nunique()

600

In [132]:
df_raw_products['active_flag'].value_counts()

active_flag
1    587
0     13
Name: count, dtype: int64

In [133]:
df_raw_products[df_raw_products['active_flag'] == 0]

,product_id,product_name,category,brand,list_price_eur,unit_cost_eur,launch_date,active_flag
42,P0043,Meridian Electronics Item 043,Electronics,Meridian,97.02,53.05,2024-01-01,0
85,P0086,Hanse Home Item 086,Home & Kitchen,Hanse,62.56,34.70,2024-01-01,0
128,P0129,Vela Sports Item 129,Sports & Outdoors,Vela,98.30,46.64,2024-01-01,0
171,P0172,Avento Beauty Item 172,Beauty & Personal Care,Avento,19.75,11.50,2024-01-01,0
214,P0215,Nova Fashion Item 215,Fashion,Nova,48.78,22.75,2024-01-01,0
257,P0258,Karo Office Item 258,Office & Stationery,Karo,29.07,19.91,2024-01-01,0
300,P0301,Luma Electronics Item 301,Electronics,Luma,191.30,96.93,2024-01-01,0
343,P0344,Nordline Home Item 344,Home & Kitchen,Nordline,74.58,43.46,2024-01-01,0
386,P0387,Meridian Sports Item 387,Sports & Outdoors,Meridian,82.99,55.67,2024-01-01,0
429,P0430,Hanse Beauty Item 430,Beauty & Personal Care,Hanse,34.38,16.42,2024-01-01,0


In [134]:
df_raw_products['launch_date'].value_counts()

launch_date
2024-01-01    600
Name: count, dtype: int64

Initial Findings — Products Dataset

**1. Product naming convention**
- Every `product_name` contains an `Item XXX` suffix that corresponds to the numerical portion of its `product_id`.
- Some products share the same base name and brand but have different product IDs and prices (e.g., `P0020` and `P0116`).
- Further clarification is needed to determine whether these represent distinct product variants or duplicate product records.

**2. Product launch dates**
- All products have the same `launch_date`.
- The meaning of this field should be clarified to determine whether it represents the actual product launch date or another event, such as a system import.

**3. Product status**
- Thirteen products have `active_flag = 0`.
- The business definition of this flag should be confirmed, particularly whether it represents the product's current status or its status at a specific point in time.

**4. Data quality**
- No missing values were identified.
- The `category` and `brand` columns contain no obvious inconsistencies.
- Product prices and unit costs contain no zero, negative, or obviously unreasonable values.

In [135]:
df_raw_order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price_eur,unit_discount_eur
0,I00000001,O0000001,P0347,1,46.69,4.67
1,I00000002,O0000001,P0002,1,73.29,7.33
2,I00000003,O0000002,P0512,1,43.24,2.16
3,I00000004,O0000002,P0341,1,22.64,1.13
4,I00000005,O0000003,P0272,2,98.41,4.92


In [136]:
df_raw_order_items.shape

(257981, 6)

In [137]:
df_raw_order_items["order_item_id"].nunique()

257981

In [138]:
df_raw_order_items["order_id"].nunique()

150000

In [139]:
df_raw_order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 257981 entries, 0 to 257980
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   order_item_id      257981 non-null  str    
 1   order_id           257981 non-null  str    
 2   product_id         257981 non-null  str    
 3   quantity           257981 non-null  int64  
 4   unit_price_eur     257798 non-null  float64
 5   unit_discount_eur  257981 non-null  float64
dtypes: float64(2), int64(1), str(3)
memory usage: 11.8 MB


In [140]:
df_raw_order_items[["quantity", "unit_price_eur", "unit_discount_eur"]].describe()

,quantity,unit_price_eur,unit_discount_eur
count,257981.000000,257798.000000,257981.000000
mean,1.199321,71.510633,3.090575
std,0.467909,45.527288,5.194779
min,1.000000,0.000000,0.000000
25%,1.000000,36.160000,0.000000
50%,1.000000,63.210000,0.000000
75%,1.000000,96.030000,4.480000
max,3.000000,219.540000,43.780000


In [144]:
df_raw_order_items["quantity"].value_counts()

quantity
1    214215
2     36111
3      7655
Name: count, dtype: int64

In [141]:
df_raw_order_items.isna().sum()

order_item_id          0
order_id               0
product_id             0
quantity               0
unit_price_eur       183
unit_discount_eur      0
dtype: int64

In [ ]:
df_raw_order_items[df_raw_order_items.isna().any(axis=1)].sample()

160

In [145]:
df_raw_order_items[df_raw_order_items["product_id"] == "P0245"]

,order_item_id,order_id,product_id,quantity,unit_price_eur,unit_discount_eur
366,I00000367,O0000219,P0245,1,80.52,0.00
534,I00000535,O0000319,P0245,2,75.59,0.00
1708,I00001709,O0001016,P0245,2,74.15,3.71
1790,I00001791,O0001064,P0245,1,73.48,3.67
1883,I00001884,O0001123,P0245,1,75.54,3.78
...,...,...,...,...,...,...
255837,I00255838,O0148787,P0245,1,73.95,0.00
256594,I00256595,O0149202,P0245,1,80.22,4.01
256862,I00256863,O0149352,P0245,1,79.84,7.98
257885,I00257886,O0149940,P0245,1,74.40,0.00


In [143]:
df_raw_order_items['product_id'].nunique()

600

Initial Findings — Order Items Dataset

**1. Record uniqueness and referential integrity**
- No duplicated `order_item_id` values were identified.
- The dataset contains 150,000 unique `order_id` values and 600 unique `product_id` values.
- Referential integrity with the orders and products datasets has not yet been verified.

**2. Quantity and discount validation**
- The `quantity` column contains only values of 1, 2, and 3, with no zero or negative quantities.
- Discount values fall within a reasonable range, with no obvious anomalies identified.

**3. Missing unit prices**
- Some records contain missing values in `unit_price_eur`.
- Missing values appear across different products, without an obvious concentration.
- Further investigation is required to determine whether missing prices are associated with specific dates, order statuses, or other characteristics after joining the orders dataset.

**4. Transaction prices versus catalog prices**
- The same product appears with different unit prices, even when the recorded discount is identical or zero.
- Transaction prices also differ from the prices recorded in the products dataset.
- Clarification is needed to determine whether these differences reflect historical pricing, promotional policies, or data-quality issues.

**5. Open business questions**
- Does `unit_price_eur` represent the actual selling price at the time of purchase?
- Does the price in `products.csv` represent the current catalog price or a historical reference price?
- How should transactions with missing unit prices be handled when calculating revenue?

In [153]:
df_raw_website_sessions.shape

(863213, 8)

In [155]:
df_raw_website_sessions.info()

<class 'pandas.DataFrame'>
RangeIndex: 863213 entries, 0 to 863212
Data columns (total 8 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   session_id                863213 non-null  str  
 1   session_date              863213 non-null  str  
 2   channel                   863213 non-null  str  
 3   device                    863213 non-null  str  
 4   region                    863213 non-null  str  
 5   pages_viewed              863213 non-null  int64
 6   session_duration_seconds  863213 non-null  int64
 7   visitor_type              863213 non-null  str  
dtypes: int64(2), str(6)
memory usage: 52.7 MB


In [156]:
df_raw_website_sessions.head()

,session_id,session_date,channel,device,region,pages_viewed,session_duration_seconds,visitor_type
0,S00000001,2025-01-01,organic_search,mobile,Bavaria,3,496,new
1,S00000002,2025-01-01,organic_search,mobile,Lower Saxony,1,191,returning
2,S00000003,2025-01-01,organic_search,desktop,Lower Saxony,4,295,returning
3,S00000004,2025-01-01,organic_search,desktop,North Rhine-Westphalia,6,364,new
4,S00000005,2025-01-01,organic_search,desktop,Hamburg,5,521,returning


In [157]:
df_raw_website_sessions["session_id"].nunique()

863013

In [162]:
df_raw_website_sessions[df_raw_website_sessions.duplicated(keep=False)].sort_values(by="session_id")

,session_id,session_date,channel,device,region,pages_viewed,session_duration_seconds,visitor_type
5005,S00005006,2025-01-03,organic_search,mobile,North Rhine-Westphalia,5,304,returning
863014,S00005006,2025-01-03,organic_search,mobile,North Rhine-Westphalia,5,304,returning
5771,S00005772,2025-01-03,paid_search,mobile,Hamburg,5,212,returning
863100,S00005772,2025-01-03,paid_search,mobile,Hamburg,5,212,returning
6129,S00006130,2025-01-03,paid_social,desktop,Hesse,1,398,new
...,...,...,...,...,...,...,...,...
863075,S00846217,2025-12-24,email,desktop,Berlin,6,419,new
863164,S00855701,2025-12-28,referral,desktop,Baden-Württemberg,4,465,new
855700,S00855701,2025-12-28,referral,desktop,Baden-Württemberg,4,465,new
863184,S00859195,2025-12-30,paid_search,mobile,Baden-Württemberg,3,138,new


In [158]:
df_raw_website_sessions["session_date"].min()

'2025-01-01'

In [159]:
df_raw_website_sessions["session_date"].max()

'2025-12-31'

In [163]:
df_raw_website_sessions["channel"].value_counts()

channel
paid_search       215964
organic_search    190707
direct            146792
paid_social       145296
email             112587
referral           51867
Name: count, dtype: int64

In [164]:
df_raw_website_sessions["device"].value_counts()

device
mobile     500745
desktop    310864
tablet      51604
Name: count, dtype: int64

In [165]:
df_raw_website_sessions["region"].value_counts()

region
Baden-Württemberg         108562
Hamburg                   107969
North Rhine-Westphalia    107960
Berlin                    107934
Hesse                     107849
Saxony                    107713
Lower Saxony              107686
Bavaria                   107540
Name: count, dtype: int64

In [167]:
df_raw_website_sessions["visitor_type"].value_counts()

visitor_type
new          431905
returning    431308
Name: count, dtype: int64

In [168]:
df_raw_website_sessions.info()

<class 'pandas.DataFrame'>
RangeIndex: 863213 entries, 0 to 863212
Data columns (total 8 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   session_id                863213 non-null  str  
 1   session_date              863213 non-null  str  
 2   channel                   863213 non-null  str  
 3   device                    863213 non-null  str  
 4   region                    863213 non-null  str  
 5   pages_viewed              863213 non-null  int64
 6   session_duration_seconds  863213 non-null  int64
 7   visitor_type              863213 non-null  str  
dtypes: int64(2), str(6)
memory usage: 52.7 MB


In [169]:
df_raw_website_sessions[["pages_viewed", "session_duration_seconds"]].describe()

,pages_viewed,session_duration_seconds
count,863213.000000,863213.000000
mean,3.498680,274.673482
std,1.707394,147.500911
min,1.000000,20.000000
25%,2.000000,147.000000
50%,3.000000,275.000000
75%,5.000000,402.000000
max,6.000000,530.000000


In [ ]:
Initial Findings — Website Sessions Dataset

**1. Duplicate records**
- Exact duplicate rows were identified, including identical `session_id` values.
- No clear explanation for these duplicates was identified.
- These records are candidates for deduplication during data cleaning to prevent inflated session counts and distorted website performance metrics.

**2. Missing values**
- No missing values were identified.

**3. Date validation**
- The dataset covers approximately one year, with no obvious date anomalies identified.

**4. Numerical validation**
- Numerical values appear reasonable, with no obvious anomalies identified.

**5. Categorical validation**
- No unexpected categories or obvious inconsistencies were identified.